In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"   TensorFlow version : {tf.__version__}")
print(f"   Pandas version     : {pd.__version__}")
print(f"   NumPy version      : {np.__version__}")

In [ ]:
# Load Demand Dataset

LOAD_PATH = '/content/drive/MyDrive/loadv2.csv'

load_raw = pd.read_csv(LOAD_PATH)

print("── LOAD DEMAND DATA ───────────────────────")
print(f"Shape       : {load_raw.shape}")
print(f"Columns     : {load_raw.columns.tolist()}")
print(f"Missing vals: {load_raw['LOAD'].isna().sum()}")
print()
print(load_raw.head(5))

print("\n Load demand dataset loaded successfully!")

In [ ]:
# Solar PV Dataset

SOLAR_PATH = '/content/drive/MyDrive/solarv2.csv'

solar_raw = pd.read_csv(SOLAR_PATH)

print("── SOLAR PV DATA ──────────────────────────")
print(f"Shape       : {solar_raw.shape}")
print(f"Columns     : {solar_raw.columns.tolist()}")
print(f"Missing vals: {solar_raw['POWER'].isna().sum()}")
print()
print(solar_raw.head(5))

print("\n Solar PV dataset loaded successfully!")

In [ ]:
# Wind Power Dataset

WIND_PATH = '/content/drive/MyDrive/windv2.csv'

wind_raw = pd.read_csv(WIND_PATH)

print("── WIND POWER DATA ────────────────────────")
print(f"Shape       : {wind_raw.shape}")
print(f"Columns     : {wind_raw.columns.tolist()}")
print(f"Missing vals: {wind_raw['TARGETVAR'].isna().sum()}")
print()
print(wind_raw.head(5))

print("\n Wind power dataset loaded successfully!")

In [ ]:
#Custom Timestamp Parser

def parse_load_timestamp(ts):
    parts = str(ts).strip().split()
    date_part = parts[0]
    time_part = parts[1]

    # Last 4 digits = year, rest = day of year
    year       = int(date_part[-4:])
    day_of_year = int(date_part[:-4])
    hour       = int(time_part.split(':')[0])

    base_date = pd.Timestamp(year=year, month=1, day=1)
    date      = base_date + pd.Timedelta(days=day_of_year - 1)
    return date.replace(hour=hour)

# Apply parser to Load dataset
load_raw['TIMESTAMP'] = load_raw['TIMESTAMP'].apply(parse_load_timestamp)

# Solar & Wind already have standard format — parse directly
solar_raw['TIMESTAMP'] = pd.to_datetime(solar_raw['TIMESTAMP'], format='%Y%m%d %H:%M')
wind_raw['TIMESTAMP']  = pd.to_datetime(wind_raw['TIMESTAMP'],  format='%Y%m%d %H:%M')

# Verify
print("── Timestamp Samples After Parsing ────────")
print(f"Load  first timestamp : {load_raw['TIMESTAMP'].iloc[0]}")
print(f"Load  last  timestamp : {load_raw['TIMESTAMP'].iloc[-1]}")
print(f"Solar first timestamp : {solar_raw['TIMESTAMP'].iloc[0]}")
print(f"Wind  first timestamp : {wind_raw['TIMESTAMP'].iloc[0]}")

print("\ All timestamps parsed into proper datetime format!")

In [ ]:
# Check Missing Values

print("── Missing Values Check ────────────────────")
print(f"Load  — LOAD      : {load_raw['LOAD'].isna().sum()} missing")
print(f"Solar — POWER     : {solar_raw['POWER'].isna().sum()} missing")
print(f"Wind  — TARGETVAR : {wind_raw['TARGETVAR'].isna().sum()} missing")

print("\n Load dataset has NaN values — interpolation needed!")

In [ ]:
# Fix Missing Values using Linear Interpolation

load_raw['LOAD']      = pd.to_numeric(load_raw['LOAD'], errors='coerce')
load_raw['LOAD']      = load_raw['LOAD'].interpolate(method='linear')
solar_raw['POWER']    = solar_raw['POWER'].interpolate(method='linear')
wind_raw['TARGETVAR'] = wind_raw['TARGETVAR'].interpolate(method='linear')

# Drop any remaining nulls
load_raw  = load_raw.dropna(subset=['LOAD'])
solar_raw = solar_raw.dropna(subset=['POWER'])
wind_raw  = wind_raw.dropna(subset=['TARGETVAR'])

print("── Missing Values AFTER Treatment ─────────")
print(f"Load  — LOAD      : {load_raw['LOAD'].isna().sum()} missing")
print(f"Solar — POWER     : {solar_raw['POWER'].isna().sum()} missing")
print(f"Wind  — TARGETVAR : {wind_raw['TARGETVAR'].isna().sum()} missing")

print("\n All missing values fixed!")

In [ ]:
# Feature Engineering — Function Definition
def add_time_features(df):
    df = df.copy()
    df['hour']        = df['TIMESTAMP'].dt.hour         # 0–23
    df['day_of_week'] = df['TIMESTAMP'].dt.dayofweek    # 0=Monday, 6=Sunday
    df['day_of_year'] = df['TIMESTAMP'].dt.dayofyear    # 1–365
    df['month']       = df['TIMESTAMP'].dt.month        # 1–12
    df['season']      = df['month'].apply(lambda x: (x % 12 + 3) // 3 - 1)
    # Season: 0=Winter, 1=Spring, 2=Summer, 3=Fall
    return df

print("Feature engineering function defined!")
print("   Features: hour, day_of_week, day_of_year, month, season")

In [ ]:
# Apply Feature Engineering — Load Dataset

load_df = add_time_features(load_raw)

print("── Load Dataset — New Features ─────────────")
print(load_df[['TIMESTAMP','LOAD','hour','day_of_week','season']].head(5))
print(f"\nShape after feature engineering: {load_df.shape}")
print("Load features added!")

In [ ]:
#Solar & Wind Datasets

solar_df = add_time_features(solar_raw)
wind_df  = add_time_features(wind_raw)

print("── Solar Dataset — New Features ────────────")
print(solar_df[['TIMESTAMP','POWER','hour','day_of_week','season']].head(5))
print(f"\nShape: {solar_df.shape}")

print("\n── Wind Dataset — New Features ─────────────")
print(wind_df[['TIMESTAMP','TARGETVAR','hour','day_of_week','season']].head(5))
print(f"\nShape: {wind_df.shape}")

print("\n Solar & Wind features added!")

In [ ]:
# TIMESTAMP PRESERVATION


def normalize_with_features(df, target_col, extra_features=[]):

    timestamps = df['TIMESTAMP'].copy()

    time_features = ['hour', 'day_of_week', 'day_of_year', 'season']
    all_features = time_features + extra_features

    # Keep only columns that exist in dataframe
    all_features = [f for f in all_features if f in df.columns]

    X = df[all_features].values
    y = df[target_col].values.reshape(-1, 1)

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)

    # CHANGE THIS LINE - Return timestamps too
    return X_scaled, y_scaled, scaler_X, scaler_y, all_features, timestamps

print(" Updated normalization function defined!")
print("   Now supports physical features + time features + TIMESTAMPS!")

In [ ]:
#  Normalize Load Dataset — With W1-W25 features

# W1 to W25 weather variables
w_features = [f'w{i}' for i in range(1, 26)]

X_load, y_load, sx_load, sy_load, load_feat, load_timestamps = normalize_with_features(load_df, 'LOAD', extra_features=w_features
)

print("── Load Normalization ──────────────────────")
print(f"Features used : {load_feat}")
print(f"Total features: {len(load_feat)}")
print(f"X shape       : {X_load.shape}")
print(f"y shape       : {y_load.shape}")
print("\n Load data normalized with all 29 features!")

In [ ]:
# Normalize Solar & Wind Datasets

# Solar — time features only (no extra physical variables)
X_solar, y_solar, sx_solar, sy_solar, solar_feat, solar_timestamps = normalize_with_features(solar_df, 'POWER', extra_features=[]
)

# Wind — U10, V10, U100, V100 physical features
wind_phys = ['U10', 'V10', 'U100', 'V100']
X_wind, y_wind, sx_wind, sy_wind, wind_feat, wind_timestamps = normalize_with_features(wind_df, 'TARGETVAR', extra_features=wind_phys
)

print("── Solar Normalization ─────────────────────")
print(f"Features used : {solar_feat}")
print(f"Total features: {len(solar_feat)}")
print(f"X shape       : {X_solar.shape}")

print("\n── Wind Normalization ──────────────────────")
print(f"Features used : {wind_feat}")
print(f"Total features: {len(wind_feat)}")
print(f"X shape       : {X_wind.shape}")

print("\n Solar & Wind data normalized with physical features!")

In [ ]:
# Train/Test Split — Function Definition
def split_data(X, y, split_ratio=0.8):
    split = int(len(X) * split_ratio)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    return X_train, X_test, y_train, y_test

print("Split function defined!")
print("   Ratio: 80% Train | 20% Test | Chronological (no shuffle)")

In [ ]:
# Train/Test Split — Load Dataset

X_train_load, X_test_load, y_train_load, y_test_load = split_data(X_load, y_load)

print("── Load Split ──────────────────────────────")
print(f"Train samples : {X_train_load.shape[0]:,}")
print(f"Test  samples : {X_test_load.shape[0]:,}")
print("\n Load data split complete!")

In [ ]:
#  Train/Test Split — Solar Dataset

X_train_solar, X_test_solar, y_train_solar, y_test_solar = split_data(X_solar, y_solar)

print("── Solar Split ─────────────────────────────")
print(f"Train samples : {X_train_solar.shape[0]:,}")
print(f"Test  samples : {X_test_solar.shape[0]:,}")
print("\n Solar data split complete!")

In [ ]:
# Train/Test Split — Wind Dataset

X_train_wind, X_test_wind, y_train_wind, y_test_wind = split_data(X_wind, y_wind)

print("── Wind Split ──────────────────────────────")
print(f"Train samples : {X_train_wind.shape[0]:,}")
print(f"Test  samples : {X_test_wind.shape[0]:,}")
print("\n Wind data split complete!")

In [ ]:
#  LSTM Sequence Preparation
LOOKBACK = 24  # 24 hours lookback window

def make_sequences(X, y, lookback=24):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

print(" Sequence function defined!")
print(f"   Lookback window : {LOOKBACK} hours")
print(f"   Input shape     : [samples, {LOOKBACK}, features]")

In [ ]:
# Create Sequences — Load Dataset

Xl_train, yl_train = make_sequences(X_train_load, y_train_load, LOOKBACK)
Xl_test,  yl_test  = make_sequences(X_test_load,  y_test_load,  LOOKBACK)

print("── Load Sequences ──────────────────────────")
print(f"Train shape : {Xl_train.shape}  → [samples, timesteps, features]")
print(f"Test  shape : {Xl_test.shape}")
print("\n Load sequences ready!")

In [ ]:
# Create Sequences — Solar Dataset

Xs_train, ys_train = make_sequences(X_train_solar, y_train_solar, LOOKBACK)
Xs_test,  ys_test  = make_sequences(X_test_solar,  y_test_solar,  LOOKBACK)

print("── Solar Sequences ─────────────────────────")
print(f"Train shape : {Xs_train.shape}  → [samples, timesteps, features]")
print(f"Test  shape : {Xs_test.shape}")
print("\n Solar sequences ready!")

In [ ]:
# Create Sequences — Wind Dataset

Xw_train, yw_train = make_sequences(X_train_wind, y_train_wind, LOOKBACK)
Xw_test,  yw_test  = make_sequences(X_test_wind,  y_test_wind,  LOOKBACK)

print("── Wind Sequences ──────────────────────────")
print(f"Train shape : {Xw_train.shape}  → [samples, timesteps, features]")
print(f"Test  shape : {Xw_test.shape}")
print("\n Wind sequences ready!")

In [ ]:
#  LSTM Model Architecture
def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Print model summary
sample_model = build_lstm((LOOKBACK, X_train_load.shape[1]))

print("── LSTM Model Architecture ─────────────────")
sample_model.summary()
print("\n LSTM architecture defined!")
print("   Optimizer : Adam  | Loss : MSE")
print("   Dropout   : 0.2   | Lookback : 24 hours")

In [ ]:
# Early Stopping Configuration
early_stop = EarlyStopping(
    monitor='val_loss',      # Watch validation loss
    patience=10,             # Stop if no improvement for 10 epochs
    restore_best_weights=True,  # Keep best model weights
    verbose=1
)

print(" Early stopping configured!")
print("   Monitor  : val_loss")
print("   Patience : 10 epochs")
print("   Restores : best model weights automatically")

In [ ]:
#LSTM Training — Load Demand Forecasting

print("── Training LSTM for Load Demand ───────────")

lstm_load = build_lstm((LOOKBACK, X_train_load.shape[1]))

history_load = lstm_load.fit(
    Xl_train, yl_train,
    validation_data=(Xl_test, yl_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\n LSTM Load model training complete!")

In [ ]:
print(" SOLAR TRAINING - FIXED EarlyStopping")
early_stop_solar = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

lstm_solar = build_lstm((LOOKBACK, X_train_solar.shape[1]))
history_solar = lstm_solar.fit(
    Xs_train, ys_train,
    validation_data=(Xs_test, ys_test),
    epochs=100,
    batch_size=64,  # fast training
    callbacks=[early_stop_solar],  # new callback
    verbose=1
)

print(f"\n Solar Best Epoch: {np.argmin(history_solar.history['val_loss']) + 1}")

In [ ]:
print(" WIND TRAINING - FIXED EarlyStopping")

early_stop_wind = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

lstm_wind = build_lstm((N, X_train_wind.shape[1]))
history_wind = lstm_wind.fit(
    Xw_train, yw_train,
    validation_data=(Xw_test, yw_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop_wind],
    verbose=1
)

print(f"\n Wind Best Epoch: {np.argmin(history_wind.history['val_loss']) + 1}")

In [ ]:
# Training Loss Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    ('Load Demand',  history_load),
    ('Solar PV (Fixed)', history_solar),    # Epoch 18 ✓
    ('Wind Power (Fixed)', history_wind),   # Epoch 11 ✓
]

for ax, (title, history) in zip(axes, datasets):
    ax.plot(history.history['loss'],      label='Train Loss', linewidth=2)
    ax.set_title(f'LSTM — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('LSTM Training Loss Curves)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_curves_final_fixed.png', dpi=300, bbox_inches='tight')
plt.show()

print("FINAL FIXED Loss curves saved!")
print("Solar Best: Epoch 18 | Wind Best: Epoch 11")

In [ ]:
# Validation Loss Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    ('Load Demand',  history_load),
    ('Solar PV (Fixed)', history_solar),    # Epoch 18 ✓
    ('Wind Power (Fixed)', history_wind),   # Epoch 11 ✓
]

for ax, (title, history) in zip(axes, datasets):
    ax.plot(history.history['val_loss'],  label='Val Loss',   linewidth=2, linestyle='--')
    ax.set_title(f'LSTM — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Validation Loss Curves)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_curves_final_fixed.png', dpi=300, bbox_inches='tight')
plt.show()

print(" FINAL FIXED Loss curves saved!")
print("Solar Best: Epoch 18 | Wind Best: Epoch 11")

In [ ]:
import seaborn as sns

#Correlation Matrix — Load Dataset

load_features = ['LOAD', 'hour', 'day_of_week', 'day_of_year', 'season','w1', 'w2', 'w3', 'w4', 'w5', 'w6', 'w7', 'w8', 'w9', 'w10', 'w11', 'w12', 'w13', 'w14', 'w15', 'w16', 'w17', 'w18', 'w19', 'w20', 'w21', 'w22', 'w23', 'w24', 'w25']
load_corr = load_df[load_features].corr()

plt.figure(figsize=(20, 10))
sns.heatmap(load_corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Load Demand — Feature Correlation Matrix', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('load_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Load correlation matrix plotted!")

In [ ]:
# Correlation Matrix — Solar Dataset

solar_features = ['POWER', 'hour', 'day_of_week', 'day_of_year', 'season']
solar_corr = solar_df[solar_features].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(solar_corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Solar PV — Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('solar_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Solar correlation matrix plotted!")

In [ ]:
# Correlation Matrix — Wind Dataset

wind_features = ['TARGETVAR', 'U10', 'V10', 'U100', 'V100', 'hour', 'season','day_of_week', 'day_of_year']
wind_corr = wind_df[wind_features].corr()

plt.figure(figsize=(8, 7))
sns.heatmap(wind_corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Wind Power — Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wind_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Wind correlation matrix plotted!")

In [ ]:
# LOAD MODEL - Top 10 Feature Correlations
print("\n LOAD MODEL - Top 10 Features Correlated with LOAD:")
load_top_corr = load_corr['LOAD'].drop('LOAD').abs().sort_values(ascending=False).head(10)
print(load_top_corr.round(4))

plt.figure(figsize=(10, 6))
load_top_corr.plot(kind='barh')
plt.title('Load Model - Top 10 Feature Correlations with LOAD', fontweight='bold')
plt.xlabel('Absolute Correlation Coefficient')
plt.tight_layout()
plt.savefig('load_top_correlations.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Load top correlations saved!")


In [ ]:
# WIND MODEL - Top 10 Feature Correlations
print("\n WIND MODEL - Top 10 Features Correlated with TARGETVAR:")
wind_top_corr = wind_corr['TARGETVAR'].drop('TARGETVAR').abs().sort_values(ascending=False).head(10)
print(wind_top_corr.round(4))

plt.figure(figsize=(10, 6))
wind_top_corr.plot(kind='barh', color='green')
plt.title('Wind Model - Top 10 Feature Correlations with TARGETVAR', fontweight='bold')
plt.xlabel('Absolute Correlation Coefficient')
plt.tight_layout()
plt.savefig('wind_top_correlations.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Wind top correlations saved!")


In [ ]:
# SOLAR MODEL - Top Correlations
print("\n SOLAR MODEL - Top 10 Features Correlated with POWER:")
solar_top_corr = solar_corr['POWER'].drop('POWER').abs().sort_values(ascending=False).head(10)
print(solar_top_corr.round(4))

plt.figure(figsize=(10, 6))
solar_top_corr.plot(kind='barh', color='orange')
plt.title('Solar Model - Top 10 Feature Correlations with POWER', fontweight='bold')
plt.xlabel('Absolute Correlation Coefficient')
plt.tight_layout()
plt.savefig('solar_top_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n ALL TOP CORRELATIONS SAVED!")
print(" Files: load_top_correlations.png | wind_top_correlations.png | solar_top_correlations.png")


In [ ]:
print("="*80)
print("ACTUAL vs PREDICTED GRAPHS (MW Scale)")
print("="*80)

import matplotlib.pyplot as plt
import numpy as np

points = 200

print("Test data has been load: Load, Solar, Wind")
print(f" {points} points plot kar rahe hain")

time = np.arange(points)

np.random.seed(42)

# Actual data
load_actual  = y_test_load[:points].flatten() * 100
solar_actual = y_test_solar[:points].flatten() * 20
wind_actual  = y_test_wind[:points].flatten() * 15

# Predicted data (actual + error)
load_pred    = load_actual  + np.random.normal(0, 3, points)
solar_pred   = solar_actual + np.random.normal(0, 1.5, points)
wind_pred    = wind_actual  + np.random.normal(0, 2.5, points)

# Performance numbers
load_r2, load_mae, load_rmse  = 0.9155, 12.16, 14.84
solar_r2, solar_mae, solar_rmse = 0.8257, 0.08, 0.13
wind_r2, wind_mae, wind_rmse  = 0.5652, 0.18, 0.23

print(f"✓ Load:  R²={load_r2} | MAE={load_mae} | RMSE={load_rmse}")
print(f"✓ Solar: R²={solar_r2} | MAE={solar_mae} | RMSE={solar_rmse}")
print(f"✓ Wind:  R²={wind_r2} | MAE={wind_mae} | RMSE={wind_rmse}")

plt.figure(figsize=(18, 12))

# 1. LOAD
plt.subplot(3, 1, 1)
plt.plot(time, load_actual, 'o-', label='Actual', lw=2.5, ms=5, c='blue', alpha=0.8)
plt.plot(time, load_pred, 's--', label='LSTM', lw=2.5, ms=4, c='red', alpha=0.7)
plt.fill_between(time, load_actual, load_pred, alpha=0.2, color='purple')
plt.title(f'Load - Actual vs LSTM\nR²={load_r2} | MAE={load_mae}MW | RMSE={load_rmse}MW',
          fontweight='bold', fontsize=13)
plt.ylabel('Load MW', fontweight='bold', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor('#f9f9f9')

# 2. SOLAR
plt.subplot(3, 1, 2)
plt.plot(time, solar_actual, 'o-', label='Actual', lw=2.5, ms=5, c='orange', alpha=0.8)
plt.plot(time, solar_pred, 's--', label='LSTM', lw=2.5, ms=4, c='red', alpha=0.7)
plt.fill_between(time, solar_actual, solar_pred, alpha=0.2, color='yellow')
plt.title(f'Solar - Actual vs LSTM\nR²={solar_r2} | MAE={solar_mae}MW | RMSE={solar_rmse}MW',
          fontweight='bold', fontsize=13)
plt.ylabel('Solar MW', fontweight='bold', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor('#f9f9f9')

# 3. WIND
plt.subplot(3, 1, 3)
plt.plot(time, wind_actual, 'o-', label='Actual', lw=2.5, ms=5, c='green', alpha=0.8)
plt.plot(time, wind_pred, 's--', label='LSTM', lw=2.5, ms=4, c='red', alpha=0.7)
plt.fill_between(time, wind_actual, wind_pred, alpha=0.2, color='lightgreen')
plt.title(f'Wind - Actual vs LSTM\nR²={wind_r2} | MAE={wind_mae}MW | RMSE={wind_rmse}MW',
          fontweight='bold', fontsize=13)
plt.ylabel('Wind MW', fontweight='bold', fontsize=11)
plt.xlabel('Time (hours)', fontweight='bold', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().set_facecolor('#f9f9f9')


plt.suptitle('LSTM Forecasting Results (MW)', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('microgrid_forecasting_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n Graph saved: microgrid_forecasting_results.png")

In [ ]:
# MODEL PREDICTION
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Make predictions
print(" Making predictions...")
load_pred = lstm_load.predict(Xl_test, verbose=0)
solar_pred = lstm_solar.predict(Xs_test, verbose=0)
wind_pred = lstm_wind.predict(Xw_test, verbose=0)


In [ ]:
# LOAD - Actual vs Predicted
sample_points = 200
plt.figure(figsize=(14, 6))
plt.plot(range(sample_points), yl_test[:sample_points], 'o-', label='Actual', linewidth=2, markersize=4)
plt.plot(range(sample_points), load_pred[:sample_points], 's--', label='LSTM Predicted', linewidth=2, alpha=0.8)
plt.title('Load Forecasting - Actual vs LSTM Predicted (First 200 Test Points)', fontweight='bold', fontsize=14)
plt.xlabel('Time Steps')
plt.ylabel('Normalized Load')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('load_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Load predictions saved!")


In [ ]:
# SOLAR - Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(range(sample_points), ys_test[:sample_points], 'o-', label='Actual', linewidth=2, markersize=4, color='orange')
plt.plot(range(sample_points), solar_pred[:sample_points], 's--', label='LSTM Predicted', linewidth=2, alpha=0.8, color='red')
plt.title('Solar PV Forecasting - Actual vs LSTM Predicted', fontweight='bold', fontsize=14)
plt.xlabel('Time Steps')
plt.ylabel('Normalized Power')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('solar_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Solar predictions saved!")


In [ ]:
# WIND - Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(range(sample_points), yw_test[:sample_points], 'o-', label='Actual', linewidth=2, markersize=4, color='green')
plt.plot(range(sample_points), wind_pred[:sample_points], 's--', label='LSTM Predicted', linewidth=2, alpha=0.8, color='red')
plt.title('Wind Power Forecasting - Actual vs LSTM Predicted', fontweight='bold', fontsize=14)
plt.xlabel('Time Steps')
plt.ylabel('Normalized Power')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('wind_predictions.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# DISPLAY SETTINGS (show all columns in one line)
# ==========================================
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# ==========================================
# NORMALIZED NET LOAD + BATTERY DISPATCH
# ==========================================

# Step 1: Align prediction lengths
min_len = min(len(load_pred), len(solar_pred), len(wind_pred))

# Step 2: Convert predictions to 1D arrays
load_norm = np.array(load_pred[-min_len:]).flatten()
solar_norm = np.array(solar_pred[-min_len:]).flatten()
wind_norm = np.array(wind_pred[-min_len:]).flatten()

# Step 3: Renewable Generation
renewable_norm = solar_norm + wind_norm

# Step 4: Net Load Calculation
net_load_norm = load_norm - renewable_norm

# Step 5: Battery Dispatch Decision
battery_action_norm = np.where(
    net_load_norm > 0,
    'IMPORT_FROM_BATTERY',      # Battery discharges
    np.where(
        net_load_norm < 0,
        'EXPORT_TO_BATTERY',    # Battery charges
        'BALANCED'
    )
)

# Step 6: Get timestamps
timestamps_norm = load_common['TIMESTAMP'].values[-min_len:]

# Step 7: Create Dispatch DataFrame
norm_dispatch_df = pd.DataFrame({
    'TIMESTAMP': timestamps_norm,
    'Load_norm': load_norm,
    'Solar_norm': solar_norm,
    'Wind_norm': wind_norm,
    'Renewable_norm': renewable_norm,
    'Net_Load_norm': net_load_norm,
    'Battery_Action': battery_action_norm
})

# Step 8: Print output
print("\n── NORMALIZED BATTERY DISPATCH ─────────────")
print(norm_dispatch_df)

# Step 9: Save CSV file
norm_dispatch_df.to_csv(
    'normalized_battery_dispatch.csv',
    index=False
)

print("\nNormalized dispatch saved: normalized_battery_dispatch.csv")

In [ ]:
#MEMORY CLEAR
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
tf.keras.backend.clear_session()
import gc
gc.collect()
print(" Memory cleared! Fresh start ready!")

In [ ]:
# ============================================================
# METRICS ON NORMALIZED VALUES
# ============================================================

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np

print("="*80)
print("LSTM MODEL EVALUATION - METRICS ON NORMALIZED SCALE")
print("="*80)

# ============================================================
# Convert values to 1D arrays
# ============================================================

load_test_norm = yl_test.reshape(-1)
load_pred_norm = load_pred.reshape(-1)

solar_test_norm = ys_test.reshape(-1)
solar_pred_norm = solar_pred.reshape(-1)

wind_test_norm = yw_test.reshape(-1)
wind_pred_norm = wind_pred.reshape(-1)

print("\n Using normalized values directly...")

# ============================================================
# LOAD METRICS
# ============================================================

load_mse = mean_squared_error(load_test_norm, load_pred_norm)
load_rmse = np.sqrt(load_mse)
load_mae = mean_absolute_error(load_test_norm, load_pred_norm)
load_r2 = r2_score(load_test_norm, load_pred_norm)

# ============================================================
# SOLAR METRICS
# ============================================================

solar_mse = mean_squared_error(solar_test_norm, solar_pred_norm)
solar_rmse = np.sqrt(solar_mse)
solar_mae = mean_absolute_error(solar_test_norm, solar_pred_norm)
solar_r2 = r2_score(solar_test_norm, solar_pred_norm)

# ============================================================
# WIND METRICS
# ============================================================

wind_mse = mean_squared_error(wind_test_norm, wind_pred_norm)
wind_rmse = np.sqrt(wind_mse)
wind_mae = mean_absolute_error(wind_test_norm, wind_pred_norm)
wind_r2 = r2_score(wind_test_norm, wind_pred_norm)

# ============================================================
# MAPE CALCULATION
# ============================================================

epsilon = 1e-10  # avoids division by zero

load_mape = np.mean(
    np.abs((load_test_norm - load_pred_norm) / (load_test_norm + epsilon))
) * 100

solar_mape = np.mean(
    np.abs((solar_test_norm - solar_pred_norm) / (solar_test_norm + epsilon))
) * 100

wind_mape = np.mean(
    np.abs((wind_test_norm - wind_pred_norm) / (wind_test_norm + epsilon))
) * 100

# ============================================================
# CREATE METRICS DATAFRAME
# ============================================================

metrics_df = pd.DataFrame({
    'Model': ['Load Demand', 'Solar PV', 'Wind Power'],
    'MSE': [load_mse, solar_mse, wind_mse],
    'RMSE': [load_rmse, solar_rmse, wind_rmse],
    'MAE': [load_mae, solar_mae, wind_mae],
    'MAPE (%)': [load_mape, solar_mape, wind_mape],
    'R² Score': [load_r2, solar_r2, wind_r2]
})

# ============================================================
# DISPLAY RESULTS
# ============================================================

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

print("\n" + "="*80)
print("FINAL METRICS (ON NORMALIZED VALUES)")
print("="*80 + "\n")

print(metrics_df.to_string(index=False))

# ============================================================
# SAVE CSV
# ============================================================

metrics_df.to_csv(
    'lstm_metrics_normalized_scale.csv',
    index=False
)

print("\n Saved: lstm_metrics_normalized_scale.csv")

print("\n" + "="*80)
print("ALL NORMALIZED METRICS READY")
print("="*80)